# Audio Retrieval — MSCLAP: Audio Queries × Text Corpus

**Uses `microsoft/msclap` to embed both audio queries and the text corpus in a shared latent space.**

| Component | Method |
|-----------|--------|
| Query Encoding | `CLAP.get_audio_embeddings(wav_paths)` |
| Corpus Encoding | `CLAP.get_text_embeddings(texts)` |

**Dataset:** `MedRAG/pubmed` (HuggingFace)  
**Metrics:** MRR, MAP, NDCG, Recall@1/5/10

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install -q msclap

In [ ]:
pip install numpy==1.26.4

In [ ]:
!pip install transformers tokenizers huggingface-hub

In [ ]:
# ── Cell 2: Config — change only this cell ────────────────────────────────
CONFIG = {
    # MSCLAP model version: "2023" or "clapcap"
    "clap_version": "2023",

    # How many corpus docs to use (None = full dataset, use small number to test)
    "corpus_size": 500,

    # Number of queries to evaluate (sampled from corpus titles)
    "num_queries": 200,

    # Retrieval depth
    "top_k": 10,

    # TTS for generating audio queries (piper-tts)
    "tts_model": "en_US-lessac-low",

    # SNR for optional noise augmentation (dB); set to None to skip noise
    "snr_db": None,

    # Output directory
    "output_dir": "/kaggle/working/results",

    # Audio directory
    "audio_dir": "/kaggle/working/audio",
}

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import io
import json
import os
import random
import wave
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from msclap import CLAP
from tqdm.notebook import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {DEVICE}")
print(f"CLAP version: {CONFIG['clap_version']}")

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["audio_dir"], exist_ok=True)

In [ ]:
# ── Cell 4: Load dataset ──────────────────────────────────────────────────
print("Loading MedRAG/pubmed (streaming mode)...")
n = CONFIG["corpus_size"] or 1000

dataset = load_dataset("MedRAG/pubmed", split="train", streaming=True)
dataset = list(dataset.take(n))

print(f"Corpus size : {len(dataset):,} documents")
print(f"Columns     : {list(dataset[0].keys())}\n")
print("Sample document:")
for k, v in dataset[0].items():
    print(f"  {k}: {str(v)[:120]}")

In [ ]:
# ── Cell 5: Build corpus, queries, qrels ─────────────────────────────────
#
# corpus  : { doc_id -> {title, content, contents} }
# queries : { qid    -> query_text }   <- title used as pseudo-query
# qrels   : { qid    -> {doc_id: 1} }  <- gold = the doc the title came from

corpus = {}
for row in dataset:
    doc_id   = str(row["id"])
    title    = (row.get("title")    or "").strip()
    content  = (row.get("content")  or "").strip()
    contents = (row.get("contents") or f"{title} {content}").strip()
    corpus[doc_id] = {
        "title"   : title,
        "content" : content,
        "contents": contents,
    }

all_doc_ids = list(corpus.keys())
random.seed(42)
query_doc_ids = random.sample(all_doc_ids, min(CONFIG["num_queries"], len(all_doc_ids)))

queries = {}
qrels   = {}
for doc_id in query_doc_ids:
    qid = f"q_{doc_id}"
    queries[qid] = corpus[doc_id]["title"]
    qrels[qid]   = {doc_id: 1}

print(f"Corpus docs : {len(corpus):,}")
print(f"Queries     : {len(queries):,}")
print(f"Qrels       : {sum(len(v) for v in qrels.values()):,} judgments")

In [ ]:
# ── Cell 6: TTS — title text → WAV audio (Piper) ─────────────────────────
!pip install -q piper-tts
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/low/en_US-lessac-low.onnx.json

from piper import PiperVoice

voice = PiperVoice.load("en_US-lessac-low.onnx", use_cuda=(DEVICE == "cuda"))


def add_noise(audio_int16: np.ndarray, snr_db: float) -> np.ndarray:
    """Add Gaussian noise to int16 PCM audio at a given SNR."""
    audio_f = audio_int16.astype(np.float32)
    signal_power = np.mean(audio_f ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), audio_f.shape)
    return np.clip(audio_f + noise, -32768, 32767).astype(np.int16)


def synthesize_and_save(text: str, qid: str) -> str:
    path = os.path.join(CONFIG["audio_dir"], f"{qid}.wav")
    if os.path.exists(path):
        return path

    buf = io.BytesIO()
    with wave.open(buf, "wb") as wav_buf:
        voice.synthesize_wav(text, wav_buf)

    buf.seek(0)
    with wave.open(buf, "rb") as wav_buf:
        params = wav_buf.getparams()
        frames = wav_buf.readframes(wav_buf.getnframes())

    audio = np.frombuffer(frames, dtype=np.int16).copy()

    if CONFIG["snr_db"] is not None:
        audio = add_noise(audio, snr_db=CONFIG["snr_db"])

    with wave.open(path, "wb") as wav_out:
        wav_out.setparams(params)
        wav_out.writeframes(audio.tobytes())

    return path


print(f"Generating audio for {len(queries):,} queries...")
query_audio = {}
for qid, text in tqdm(queries.items(), desc="TTS"):
    query_audio[qid] = synthesize_and_save(text, qid)

print(f"Done. Sample: {list(query_audio.items())[0]}")

In [ ]:
# ── Cell 7: Load MSCLAP ───────────────────────────────────────────────────
# use_cuda=True automatically if GPU is available
clap_model = CLAP(version=CONFIG["clap_version"], use_cuda=(DEVICE == "cuda"))
print(f"CLAP model loaded (version={CONFIG['clap_version']})")

In [ ]:
# ── Cell 8: Encode corpus with CLAP text encoder ─────────────────────────
# CLAP.get_text_embeddings expects a list of strings; returns a torch Tensor

def encode_texts(texts, desc="Encoding text"):
    all_embs = []
    # Process 1-by-1 to avoid MSCLAP's unpadded batching error
    for text in tqdm(texts, desc=desc):
        # Truncate text to a safe length (~300 words) to avoid exceeding 
        # the transformer's 512 max token limit.
        truncated_text = " ".join(text.split()[:300]) 
        
        with torch.no_grad():
            # Pass as a single-item list
            embs = clap_model.get_text_embeddings([truncated_text])  # (1, D)
        all_embs.append(embs.cpu().numpy())
        
    return np.vstack(all_embs)

doc_ids   = list(corpus.keys())
doc_texts = [corpus[d]["content"] for d in doc_ids]

corpus_embeddings = encode_texts(doc_texts, desc="Encoding corpus")
# L2-normalise for cosine similarity via dot product
corpus_embeddings = corpus_embeddings / (np.linalg.norm(corpus_embeddings, axis=1, keepdims=True) + 1e-9)
print(f"Corpus embeddings : {corpus_embeddings.shape}")

In [ ]:
# ── Cell 9: Encode audio queries with CLAP audio encoder ─────────────────
# CLAP.get_audio_embeddings expects a list of file paths; returns a torch Tensor

BATCH = 64

def encode_audio(wav_paths, desc="Encoding audio"):
    all_embs = []
    for start in tqdm(range(0, len(wav_paths), BATCH), desc=desc):
        batch = wav_paths[start : start + BATCH]
        with torch.no_grad():
            embs = clap_model.get_audio_embeddings(batch, resample=True)  # (B, D)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)


qids      = list(queries.keys())
wav_paths = [query_audio[q] for q in qids]

query_embeddings = encode_audio(wav_paths, desc="Encoding audio queries")
query_embeddings = query_embeddings / (np.linalg.norm(query_embeddings, axis=1, keepdims=True) + 1e-9)
print(f"Query embeddings  : {query_embeddings.shape}")

In [ ]:
# ── Cell 10: Retrieval (cosine sim via dot product on L2-normed vecs) ─────
print("Running retrieval ...")

scores      = query_embeddings @ corpus_embeddings.T  # (num_queries, num_docs)
top_k       = CONFIG["top_k"]
ranked_idxs = np.argsort(-scores, axis=1)[:, :top_k]

results = {}
for i, qid in enumerate(qids):
    results[qid] = [
        (doc_ids[idx], float(scores[i, idx]))
        for idx in ranked_idxs[i]
    ]

print(f"Retrieval done for {len(results):,} queries, top-{top_k} docs each.")

In [ ]:
# ── Cell 11: Evaluation ───────────────────────────────────────────────────
def compute_metrics(results, qrels, top_k=10):
    """
    Compute MRR, MAP, NDCG@k, Recall@1/5/10.

    Args:
        results : { qid -> [(doc_id, score), ...] }  ranked list
        qrels   : { qid -> {doc_id: relevance} }
        top_k   : cutoff for NDCG and Recall

    Returns:
        dict of metric_name -> float
    """
    mrr_scores, map_scores, ndcg_scores = [], [], []
    recall_at = {1: [], 5: [], 10: []}

    for qid, ranked in results.items():
        relevant = set(qrels.get(qid, {}).keys())
        if not relevant:
            continue

        ranked_ids = [doc_id for doc_id, _ in ranked]

        # ── MRR ──────────────────────────────────────────────────────────
        rr = 0.0
        for rank, doc_id in enumerate(ranked_ids, start=1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrr_scores.append(rr)

        # ── MAP ──────────────────────────────────────────────────────────
        num_relevant_seen, precision_sum = 0, 0.0
        for rank, doc_id in enumerate(ranked_ids, start=1):
            if doc_id in relevant:
                num_relevant_seen += 1
                precision_sum += num_relevant_seen / rank
        ap = precision_sum / len(relevant) if relevant else 0.0
        map_scores.append(ap)

        # ── NDCG@k ───────────────────────────────────────────────────────
        def dcg(ids, k):
            return sum(
                (qrels[qid].get(doc_id, 0)) / np.log2(rank + 1)
                for rank, doc_id in enumerate(ids[:k], start=1)
            )
        ideal_ids = sorted(relevant, key=lambda d: qrels[qid].get(d, 0), reverse=True)
        idcg = dcg(ideal_ids, top_k)
        ndcg = dcg(ranked_ids, top_k) / idcg if idcg > 0 else 0.0
        ndcg_scores.append(ndcg)

        # ── Recall@k ─────────────────────────────────────────────────────
        for k in recall_at:
            hits = len(set(ranked_ids[:k]) & relevant)
            recall_at[k].append(hits / len(relevant))

    metrics = {
        "MRR"          : float(np.mean(mrr_scores)),
        "MAP"          : float(np.mean(map_scores)),
        f"NDCG@{top_k}": float(np.mean(ndcg_scores)),
        "Recall@1"     : float(np.mean(recall_at[1])),
        "Recall@5"     : float(np.mean(recall_at[5])),
        "Recall@10"    : float(np.mean(recall_at[10])),
    }
    return metrics


metrics = compute_metrics(results, qrels, top_k=CONFIG["top_k"])

print("\n" + "="*50)
print(f" MSCLAP (v{CONFIG['clap_version']}) — Audio → Text Retrieval")
print("="*50)
for name, val in metrics.items():
    print(f"  {name:<15} {val:.4f}")
print("="*50)

In [ ]:
# ── Cell 12: Save results ─────────────────────────────────────────────────
output = {
    "stage"       : "MS Clap",
    "clap_version": CONFIG["clap_version"],
    "corpus_size" : len(corpus),
    "num_queries" : len(queries),
    "top_k"       : CONFIG["top_k"],
    "snr_db"      : CONFIG["snr_db"],
    "metrics"     : metrics,
}

out_path = Path(CONFIG["output_dir"]) / f"msclap_{CONFIG['clap_version']}_{CONFIG['corpus_size']}.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to: {out_path}")